In [ ]:
# -*- coding: utf-8 -*-
# =====================================================================================
#  [학생용] 결과기 개발 기본 틀 — 1번 셀
# =====================================================================================
#  이 셀은 완성된 결과기가 아닙니다. 1번 셀에 팀별 결과기를 구현한 뒤 사용합니다.
#  결과기 코랩은 아래 두 셀을 위에서 아래로 한 번 실행할 수 있어야 합니다.
#
#    1번 셀: 팀별 결과기 구현 — 이 파일의 코드
#    2번 셀: 공개 10문항 공통 러너 — 운영진 배포본, 팀 식별자 한 줄 외 수정 금지
#
#  ┌─ 반드시 유지할 계약 ───────────────────────────────────────────────────────────┐
#  │ · answer_question(question: str) 함수 이름과 입력 형식                         │
#  │ · 반환값: {"answer": 문자열, "retrieved": [[문서명, 조번호], ...]}            │
#  │ · retrieved: 실제 답변에 사용한 근거를 관련도 순으로 1~4개                    │
#  │ · 전역 FastAPI app, GET /health, POST /answer                                 │
#  │ · Qwen2.5-Instruct 계열 생성 모델을 Colab T4에서 로컬 실행                    │
#  │ · 새 Colab T4 런타임에서 외부 준비 작업 없이 위에서 아래로 한 번 실행         │
#  └────────────────────────────────────────────────────────────────────────────────┘
#
#  ┌─ 팀이 자유롭게 구현할 부분 ─────────────────────────────────────────────────────┐
#  │ · 1번 셀 안의 결과기 구현 방식과 필요한 패키지                                 │
#  │ · answer_question 함수 내부의 처리 방식                                        │
#  │   단, 위의 고정 계약과 아래의 금지 조건은 유지해야 합니다.                     │
#  └────────────────────────────────────────────────────────────────────────────────┘
#
#  사용할 수 없는 방식
#    · Google Drive 마운트, 미리 업로드한 파일, 개인 컴퓨터 경로에 의존하는 코드
#    · 외부 생성형 LLM API, 원격 임베딩·리랭커, 원격 관리형 검색 서비스
#    · 실행 중 사람의 파일 업로드·문자 입력·버튼 클릭을 기다리는 코드
#    · torch 재설치, torch.compile
#    · 질문과 관계없이 약관 원문 전체를 매 질문의 프롬프트에 넣는 방식
#
#  주의
#    · 약관 원문을 확보하는 방법은 팀별 자유 구현입니다.
#    · 공개·비공개 답변 JSON은 2번 셀이 생성합니다. 1번 셀에서 직접 만들지 않습니다.
# =====================================================================================


# -------------------------------------------------------------------------------------
# 0. 고정 기준 — 문서명과 생성 모델 계열
# -------------------------------------------------------------------------------------
# retrieved에 기록하는 문서명은 아래 네 이름 중 하나를 그대로 사용합니다.
# 조번호는 3 또는 "제3조"처럼 채점기가 조번호를 식별할 수 있는 형태로 반환합니다.
OFFICIAL_DOCUMENT_NAMES = (
    "카카오계정 약관",
    "카카오 위치정보 이용약관",
    "카카오 통합서비스약관",
    "카카오 통합 약관",
)

# 정확한 모델 크기와 로딩 옵션은 자유지만 생성 모델은 이 계열을 사용합니다.
REQUIRED_GENERATION_MODEL_FAMILY = "Qwen2.5-Instruct"


# =====================================================================================
# 1. 팀별 자유 구현 영역 — 함수 내부를 팀 코드로 교체합니다
# =====================================================================================
# 이 영역에는 팀이 필요한 패키지 설치와 결과기 구현 코드를 작성합니다.
# 구현 방법을 제한하지 않으며, 운영진은 아래 answer_question 함수만 호출합니다.


def answer_question(question: str):
    """공통 러너가 질문마다 호출하는 고정 진입점입니다.

    함수 내부 구현은 자유지만 다음 반환 계약은 반드시 유지합니다.

    return {
        "answer": "질문에 대한 최종 답변",
        "retrieved": [
            ["카카오계정 약관", 3],
            ["카카오 통합서비스약관", 7],
        ],
    }

    retrieved에는 내부 검색 후보 전체가 아니라 실제 답변 생성에 사용한 핵심 근거를
    관련도 순으로 1~4개만 기록합니다.
    """
    if not isinstance(question, str) or not question.strip():
        raise ValueError("question은 비어 있지 않은 문자열이어야 합니다.")
    raise NotImplementedError(
        "answer_question()을 팀별 결과기로 구현하고, "
        "answer 문자열과 retrieved 1~4개를 고정 형식으로 반환하세요."
    )


# =====================================================================================
# 2. 고정 FastAPI 연결 영역 — 삭제하거나 경로를 바꾸지 않습니다
# =====================================================================================
# 2번 공통 러너는 아래 app을 localhost에서 실행하고 다음 주소를 호출합니다.
#   · GET  /health : 결과기 서버 준비 여부 확인
#   · POST /answer : {"question": "..."}을 보내 answer_question() 결과 수신
#
# 팀별 결과기 로직은 위 자유 구현 영역에서 작성합니다. 이 블록은 서버 연결만 담당합니다.
# 동시 요청에서 하나의 GPU 생성 모델이 충돌하지 않도록 Lock을 사용합니다.
import subprocess
import sys
import threading


def _install_server_packages():
    """공통 러너와 연결하는 데 필요한 가벼운 서버 패키지만 설치합니다."""
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "fastapi", "uvicorn"],
        check=True,
    )


_install_server_packages()

from fastapi import FastAPI, HTTPException  # noqa: E402


app = FastAPI(title="KTB AI Performance Result Generator")
_GENERATION_LOCK = threading.Lock()


@app.get("/health")
def health():
    return {"status": "ok"}


@app.post("/answer")
def answer_api(payload: dict):
    question = payload.get("question")
    if not isinstance(question, str) or not question.strip():
        raise HTTPException(status_code=400, detail="question must be a non-empty string")
    with _GENERATION_LOCK:
        return answer_question(question.strip())


print("[1번 셀 준비] 결과기 구현을 마친 뒤 2번 공통 러너를 실행하세요.")
